In [ ]:
# ── Cell 1 · Dependencies ─────────────────────────────────────────────────────
!pip install -q \
    transformers \
    accelerate \
    peft \
    bitsandbytes \
    fastapi \
    uvicorn \
    nest-asyncio \
    sentencepiece \
    safetensors

In [ ]:
# ── Cell 2 · Imports ──────────────────────────────────────────────────────────

# Standard library
import os
import re
import subprocess
import threading
import time
import uuid

# Third party
import torch
import nest_asyncio
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
from peft import PeftModel
from google.colab import drive

# FastAPI
import uvicorn
from fastapi import FastAPI, HTTPException
from pydantic import BaseModel, Field

nest_asyncio.apply()
print("Imports OK")

In [ ]:
# ── Cell 3 · Configuration ────────────────────────────────────────────────────

BASE_MODEL     = "sarvamai/sarvam-2b-v0.5"
MODEL_NAME     = "sarvam-triage-v1"
PORT           = 8001
MAX_NEW_TOKENS = 512
TEMPERATURE    = 0.3
TOP_P          = 0.9

SYSTEM_PROMPT = (
    "You are a multilingual medical triage assistant. "
    "Analyse the patient symptoms and return ONLY a valid JSON object. "
    "Do not include markdown, explanation, or any text outside the JSON."
)

print("Configuration set.")

In [ ]:
# ── Cell 4 · Load Model ───────────────────────────────────────────────────────

# 4a — Mount Drive and find the adapter automatically
drive.mount("/content/drive", force_remount=True)

ADAPTER_PATH = None
for root, dirs, files in os.walk("/content/drive/MyDrive"):
    if "adapter_config.json" in files:
        ADAPTER_PATH = root
        break

if ADAPTER_PATH is None:
    raise FileNotFoundError(
        "adapter_config.json not found anywhere in Google Drive. "
        "Make sure the LoRA folder is uploaded to MyDrive."
    )

print(f"Adapter found at: {ADAPTER_PATH}")

# 4b — Tokenizer
print("Loading tokenizer...")
tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 4c — Base model (4-bit quantised)
print("Loading base model (3-5 min on first run)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)
base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
    trust_remote_code=True,
)

# 4d — LoRA adapter
print("Applying LoRA adapter...")
model = PeftModel.from_pretrained(base_model, ADAPTER_PATH)
model.eval()

print()
print("Model loaded successfully")
print(f"  GPU : {torch.cuda.get_device_name(0)}")
print(f"  VRAM: {torch.cuda.memory_allocated() / 1e9:.2f} GB used")

In [ ]:
# ── Cell 5 · Inference Test ───────────────────────────────────────────────────

def build_prompt(user_msg: str) -> str:
    return (
        f"<|im_start|>system\n{SYSTEM_PROMPT}<|im_end|>\n"
        f"<|im_start|>user\n{user_msg}<|im_end|>\n"
        f"<|im_start|>assistant\n"
    )

def run_inference(prompt: str) -> tuple[str, int]:
    full_prompt = build_prompt(prompt)
    inputs = tokenizer(full_prompt, return_tensors="pt").to(model.device)

    t0 = time.perf_counter()
    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=True,
            temperature=TEMPERATURE,
            top_p=TOP_P,
            pad_token_id=tokenizer.pad_token_id,
            eos_token_id=tokenizer.eos_token_id,
        )
    latency_ms = int((time.perf_counter() - t0) * 1000)

    full_response = tokenizer.decode(output_ids[0], skip_special_tokens=False)
    if "<|im_start|>assistant" in full_response:
        reply = full_response.split("<|im_start|>assistant")[-1]
        reply = reply.replace("<|im_end|>", "").strip()
    else:
        new_ids = output_ids[0][inputs["input_ids"].shape[-1]:]
        reply = tokenizer.decode(new_ids, skip_special_tokens=True).strip()

    return reply, latency_ms

# Smoke test
print("Running test inference...")
output, ms = run_inference("Patient has fever for three days.")
print(f"Latency : {ms} ms")
print(f"Output  : {output}")

In [ ]:
# ── Cell 6 · FastAPI Server ───────────────────────────────────────────────────

class GenerateRequest(BaseModel):
    prompt: str = Field(..., min_length=1)
    request_id: str | None = None

class GenerateResponse(BaseModel):
    generated_text: str
    latency_ms: int
    model: str
    request_id: str

app = FastAPI(title="Sarvam Triage Inference", version="1.0.0")

@app.get("/health")
def health():
    return {"status": "healthy", "model_loaded": True, "model": MODEL_NAME}

@app.post("/generate", response_model=GenerateResponse)
def generate(payload: GenerateRequest):
    rid = payload.request_id or str(uuid.uuid4())
    try:
        text, latency_ms = run_inference(payload.prompt)
        return GenerateResponse(
            generated_text=text, latency_ms=latency_ms,
            model=MODEL_NAME, request_id=rid,
        )
    except Exception as exc:
        raise HTTPException(status_code=500, detail=str(exc)) from exc

# Start server
config = uvicorn.Config(app, host="0.0.0.0", port=PORT, log_level="warning")
server = uvicorn.Server(config)
threading.Thread(target=server.run, daemon=True).start()
time.sleep(2)
print(f"Server running on port {PORT}")

In [ ]:
# ── Cell 7 · Cloudflare Tunnel ────────────────────────────────────────────────

!wget -q https://github.com/cloudflare/cloudflared/releases/latest/download/cloudflared-linux-amd64 -O cloudflared
!chmod +x cloudflared

proc = subprocess.Popen(
    ["./cloudflared", "tunnel", "--url", f"http://localhost:{PORT}"],
    stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True,
)

public_url = None
for line in proc.stdout:
    match = re.search(r"https://[\w-]+\.trycloudflare\.com", line)
    if match:
        public_url = match.group(0)
        break

print("\n" + "=" * 60)
print(f"  URL : {public_url}")
print(f"  Add to backend .env:")
print(f"  MODEL_ENDPOINT={public_url}")
print("=" * 60)

In [ ]:
# ── Cell 8 · Keep Alive ───────────────────────────────────────────────────────
# Prevents Colab from disconnecting. Stop this cell to shut down.
print("Server live. Stop this cell to shut down.")
while True:
    time.sleep(60)